# CASMI26: FPNet-NP Data Audit

Step 1 of the FPNet-NP study. The fold-safe FPNet ranks the right structure first for 68% of new molecules but only 33%
of never-seen natural products, and the hidden test is natural products. The idea is to refit it with extra public
natural-product spectra. Its training set already holds 2.54 million spectra from GNPS, MassBank, MoNA, RIKEN, MSnLib
and other libraries, so before building anything this notebook measures how much the public libraries actually add:
structures that are **not** in the competition train set, how many of them are natural products, and how many spectra
they bring, per source.

Inputs: the competition train set and the identity-disjoint folds (Dmitrii Gluzdov), [nakhun/casmi26-extlib2](https://www.kaggle.com/datasets/nakhun/casmi26-extlib2),
[aryanawasthiteykik/casmi26-gnps-community-cc0](https://www.kaggle.com/datasets/aryanawasthiteykik/casmi26-gnps-community-cc0),
[aryanawasthiteykik/casmi26-e180-not-in-train](https://www.kaggle.com/datasets/aryanawasthiteykik/casmi26-e180-not-in-train)
and the natural-product knowledge table (Dmitrii Gluzdov). Output: `np_audit.json` and `np_new_spectra_keys.parquet`.

In [ ]:
import os, sys, glob, json, time, subprocess
T0 = time.time()
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
try:
    import rdkit
    print('rdkit already installed:', rdkit.__version__)
except ImportError:
    whl = [w for w in sorted(glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)) if f'-{tag}-' in w]
    assert whl, f'no RDKit wheel for {tag}: ' + str(glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--quiet', whl[0]], check=True)
import numpy as np, pandas as pd, pyarrow.parquet as pq
from rdkit import Chem, RDLogger, rdBase
RDLogger.DisableLog('rdApp.*')
print('rdkit', rdBase.rdkitVersion)


def find(name):
    hits = sorted(glob.glob('/kaggle/input/**/' + name, recursive=True), key=len)
    return hits[0] if hits else None


def pick(cols, names):
    low = {c.lower(): c for c in cols}
    return next((low[n] for n in names if n in low), None)


KC = {}
def k14(smi):
    if smi in KC:
        return KC[smi]
    k = ''
    try:
        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None
        if m is not None:
            k = Chem.MolToInchiKey(m)[:14]
    except Exception:
        k = ''
    KC[smi] = k
    return k

## Train structures and evaluation molecules

All train structures come from the identity-disjoint folds table (one row per structure, InChIKey-14). The natural
products that the FPNet evaluation and the panel hold out are read from the fold files when present, so that the audit
can report how many of them the public libraries contain (they would have to be excluded from any refit).

In [ ]:
fs = find('casmi26-identity-disjoint-validation-folds/structures.parquet')
st = pd.read_parquet(fs)
kcol = pick(st.columns, ['inchikey14', 'ik14', 'key14'])
TRAIN = set(st[kcol].astype(str))
print('train structures', len(TRAIN), '| columns', list(st.columns)[:12])
fold_col = pick(st.columns, ['fold_identity', 'fold'])
FOLD0 = set(st.loc[st[fold_col].astype(str).isin(['0', 'fold0']), kcol].astype(str)) if fold_col else set()
print('fold-0 structures', len(FOLD0))
PANEL = set()
pp = find('panel.parquet')
if pp:
    pan = pd.read_parquet(pp)
    pk = pick(pan.columns, ['inchikey14', 'ik14', 'key14', 'key'])
    if pk:
        PANEL = set(pan[pk].astype(str).str[:14])
print('panel molecules', len(PANEL))

## Natural-product structures

In [ ]:
NP = set(); np_src = None
for name in ('natural_products.parquet',):
    p = find(name)
    if p:
        cols = pq.read_schema(p).names
        c = pick(cols, ['inchikey14', 'ik14', 'key14', 'inchikey'])
        if c:
            NP = set(pd.read_parquet(p, columns=[c])[c].astype(str).str[:14]); np_src = p
print('natural-product structures', len(NP), 'from', np_src)

## Public libraries: rows, structures, and what is new

In [ ]:
def audit(tag, path, smi_names, key_names, extra_names=()):
    cols = pq.read_schema(path).names
    cs, ck = pick(cols, smi_names), pick(cols, key_names)
    ce = [c for c in (pick(cols, [e]) for e in extra_names) if c]
    use = [c for c in (cs, ck, *ce) if c]
    t = pq.read_table(path, columns=use).to_pandas()
    t0 = time.time()
    if ck:
        keys = t[ck].astype(str).str[:14].values
    else:
        keys = np.array([k14(s) for s in t[cs].values])
    t['_k'] = keys
    t = t[t._k.str.len() == 14]
    t['_new'] = ~t._k.isin(TRAIN)
    t['_np'] = t._k.isin(NP)
    t['_fold0'] = t._k.isin(FOLD0)
    t['_panel'] = t._k.isin(PANEL)
    r = dict(source=tag, columns=cols, rows=int(len(t)), structures=int(t._k.nunique()),
             new_rows=int(t._new.sum()), new_structures=int(t.loc[t._new, '_k'].nunique()),
             new_np_rows=int((t._new & t._np).sum()), new_np_structures=int(t.loc[t._new & t._np, '_k'].nunique()),
             np_structures_total=int(t.loc[t._np, '_k'].nunique()),
             fold0_structures=int(t.loc[t._fold0, '_k'].nunique()), panel_structures=int(t.loc[t._panel, '_k'].nunique()),
             key_seconds=round(time.time() - t0, 1))
    for c in ce:
        g = t.groupby(c).agg(rows=('_k', 'size'), new=('_new', 'sum'), new_np=('_np', lambda s: int((s & t.loc[s.index, '_new']).sum())))
        r['by_' + c] = g.sort_values('new', ascending=False).head(25).astype(int).to_dict('index')
    print(json.dumps({k: v for k, v in r.items() if not k.startswith('by_') and k != 'columns'}), flush=True)
    return r, t.loc[t._new, ['_k', '_np']].assign(source=tag)


REPORT, NEW = [], []
srcs = [('extlib2', find('ext2_meta.parquet'), ['normalized_smiles', 'smiles'], ['inchikey14', 'ik14'], ['source', 'lib', 'library']),
        ('gnps_community', find('gnps_cc0_trainschema.parquet'), ['normalized_smiles', 'smiles'], ['inchikey14', 'ik14'], ['library', 'source', 'library_name', 'libraryname']),
        ('e180_not_in_train', find('e180_not_in_train.parquet'), ['smiles', 'normalized_smiles'], ['ik14', 'inchikey14'], [])]
for tag, path, sn, kn, en in srcs:
    if not path:
        print(tag, 'not attached'); continue
    r, new = audit(tag, path, sn, kn, en)
    REPORT.append(r); NEW.append(new)

## Summary across sources

In [ ]:
allnew = pd.concat(NEW, ignore_index=True) if NEW else pd.DataFrame(columns=['_k', '_np', 'source'])
summary = dict(
    train_structures=len(TRAIN), natural_product_table=len(NP),
    union_new_structures=int(allnew._k.nunique()), union_new_np_structures=int(allnew.loc[allnew._np, '_k'].nunique()),
    union_new_rows=int(len(allnew)), union_new_np_rows=int(allnew._np.sum()),
    seconds=round(time.time() - T0, 1))
print(json.dumps(summary, indent=1))
json.dump(dict(summary=summary, sources=REPORT), open('/kaggle/working/np_audit.json', 'w'), indent=1, default=str)
allnew.rename(columns={'_k': 'inchikey14', '_np': 'is_np'}).to_parquet('/kaggle/working/np_new_spectra_keys.parquet', index=False)
print('written np_audit.json, np_new_spectra_keys.parquet')